In [1]:
import sys

In [2]:
# sys.path.append("/home/gputnam/Diffusion-Anomaly-Detection/diffusion-anomaly")
sys.path.append("anomaly-detection/diffusion-anomaly")

In [3]:
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
    add_dict_to_argparser,
    create_gaussian_diffusion
)

from guided_diffusion.resample import UniformSampler
from guided_diffusion import dist_util

from guided_diffusion.fp16_util import *

from guided_diffusion.respace import space_timesteps

import numpy as np
import matplotlib.pyplot as plt
import torch as th

import h5py

import os

import pickle

/home/munjung/.conda/envs/diffusion/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
plt.rcParams.update({'font.size': 14})

In [5]:
# imgs = np.expand_dims(np.stack([undershoot, bad_wire, dead_wires]), axis=1)
# imgs = th.tensor(imgs, device=dist_util.dev())

In [6]:
def visualize(img):
    return np.clip(img, -1, 1)

In [7]:
# Downsample this_arr by a factor of 4 (average over 4x4 blocks)
def downsample_by(arr, factor):
    h, w = arr.shape
    # Make sure dimensions are multiples of 4
    h_new = (h // factor) * factor
    arr_cropped = arr[:h_new, :]
    # Reshape and mean
    arr_down = arr_cropped.reshape(h_new//factor, factor, w, 1).mean(axis=(1, 3))
    return arr_down

In [ ]:
# # diagonal noise event
# filename = "/exp/sbnd/data/users/munjung/anomaly-detection/file35.h5"
# frames = h5py.File(filename, "r")
# data = frames.get("{}/raw".format(6))

# # shadow event
# filename = "/exp/sbnd/data/users/munjung/anomaly-detection/file50.h5"
# frames = h5py.File(filename, "r")
# data = frames.get("{}/raw".format(6))

# example_npz_file = "/scratch/7DayLifetime/munjung/ICARUS/plane0_npz/82090893_1115-bnb-nom-tpc1-plane0.npz"
# frames = np.load(example_npz_file)
# data = frames["reco"].astype(np.float32)
# data = np.array(data)[0][0]


# file_base_dir = "/scratch/7DayLifetime/munjung/ICARUS/plane0"
# outdir = "/scratch/7DayLifetime/munjung/ICARUS/plane0_npz"
# files = os.listdir(file_base_dir)
# filename = files[0]
rec_h5_path = "/scratch/7DayLifetime/munjung/ICARUS/plane1/82090893_1222-bnb-nom-tpc0-plane1.h5"
f_reco = h5py.File(rec_h5_path, 'r')
this_arr = f_reco['event_0']['deconvolved_signal'][40:552, 20:532]

plt.imshow(this_arr, vmin=-1, vmax=1, cmap="bwr", aspect="auto")
plt.colorbar()

In [ ]:
allarrs = []
nrows, ncols = (512, 512)
planeno = 2
planearr = this_arr #[:, 4250:]
# planearr = data[:, 3968:5638]
h, w = planearr.shape
planearr_cropped = planearr[:(h // nrows) * nrows, :(w // nrows) * nrows]

maxval = planearr_cropped.max()
badwire_vals = np.random.uniform(-maxval*1, maxval*1, size=planearr_cropped.shape[0])
badwire_idx = np.random.randint(0, planearr_cropped.shape[0])
print("badwire_idx", badwire_idx)
badwire_idx = 380
planearr_cropped[:, badwire_idx] = badwire_vals

# bad wire with pixel values between -maxval and +maxval in sine wave pattern
# stretch_in_time = 200
# stretch_in_space = 30
# badwire_vals = np.abs(np.sin(np.linspace(0, 2*np.pi, stretch_in_time)) * maxval) *0.07
# # badwire_idx = np.random.randint(0, planearr_cropped.shape[0])
# badwire_idx = 232
# for i in range(stretch_in_space):
#     time_lo = badwire_idx-100
#     time_hi = badwire_idx + stretch_in_time-100
#     if time_hi > planearr_cropped.shape[0]:
#         diff = time_hi - planearr_cropped.shape[0]
#         time_lo -= diff
#         time_hi -= diff
#     planearr_cropped[time_lo:time_hi, badwire_idx+i] += badwire_vals

H_crop, W_crop = planearr_cropped.shape
n_patches_h = H_crop // nrows
n_patches_w = W_crop // ncols
ll = planearr_cropped.reshape(n_patches_h, nrows, n_patches_w, ncols).swapaxes(1, 2).reshape(
    -1, nrows, ncols
)

# Use with stitch_patches(...) after inference to rebuild the cropped plane (H_crop x W_crop).
patch_layout = {
    "patch_size": (nrows, ncols),
    "grid_shape": (n_patches_h, n_patches_w),
    "cropped_shape": (H_crop, W_crop),
    "full_plane_shape": (h, w),
}

# cscale = [200., 100., 200.][planeno]
cscale = [1., 1., 1.][planeno]
allarrs.append(ll / cscale)

imgs = visualize(np.expand_dims(np.concatenate(allarrs), axis=1)).astype(np.float32)

In [ ]:
# maxval = planearr_cropped.max()
# # make a "bad wire", replacing a random wire with a random value between -maxval and +maxval

# badwire_vals = np.random.uniform(-maxval*3, maxval*3, size=planearr_cropped.shape[0])
# planearr_cropped_bad = planearr_cropped.copy()
# badwire_idx = np.random.randint(0, planearr_cropped.shape[0])
# planearr_cropped_bad[:, badwire_idx] = badwire_vals

plt.imshow(planearr_cropped, vmin=-1, vmax=1, cmap="bwr", aspect="auto")
plt.colorbar()
plt.show();

# plt.imshow(planearr_cropped_bad, vmin=-3, vmax=3, cmap="bwr", aspect="auto")
# plt.colorbar()
# plt.show();

In [ ]:
def stitch_patches(patches, grid_shape, patch_size=None):
    """Rebuild a 2D plane from row-major patches (same order as cell above).

    Parameters
    ----------
    patches : np.ndarray or th.Tensor
        Shape (N, ph, pw) or (N, C, ph, pw) with N = grid_shape[0] * grid_shape[1].
    grid_shape : tuple[int, int]
        (n_patches_h, n_patches_w); pass patch_layout['grid_shape'].
    patch_size : tuple[int, int] | None
        (ph, pw); defaults to patches.shape[-2:].

    Returns
    -------
    Stitched array/tensor: (H, W) or (C, H, W) matching patch_layout['cropped_shape'].
    """
    nh, nw = grid_shape
    if patch_size is None:
        ph, pw = int(patches.shape[-2]), int(patches.shape[-1])
    else:
        ph, pw = patch_size

    if isinstance(patches, th.Tensor):
        if patches.dim() == 4:
            n, c, _, _ = patches.shape
            x = patches.view(nh, nw, c, ph, pw).permute(2, 0, 3, 1, 4).reshape(c, nh * ph, nw * pw)
            return x
        if patches.dim() == 3:
            x = patches.view(nh, nw, ph, pw).permute(0, 2, 1, 3).reshape(nh * ph, nw * pw)
            return x
        raise ValueError(f"Expected 3D or 4D tensor, got {patches.dim()}D")

    if patches.ndim == 4:
        n, c, _, _ = patches.shape
        return patches.reshape(nh, nw, c, ph, pw).transpose(0, 2, 1, 3, 4).reshape(c, nh * ph, nw * pw)
    if patches.ndim == 3:
        return patches.reshape(nh, nw, ph, pw).swapaxes(1, 2).reshape(nh * ph, nw * pw)
    raise ValueError(f"Expected 3D or 4D array, got {patches.ndim}D")



In [ ]:
data.shape
imgs.shape

In [ ]:
for i in range(1):
    plt.figure(i)
    plt.imshow(np.squeeze(imgs[i]), vmin=-1, vmax=1, cmap="bwr", aspect="auto")

In [ ]:
imgs = th.tensor(imgs, device=dist_util.dev())
th.set_grad_enabled(False)

In [ ]:
args = model_and_diffusion_defaults()
diffusion_args = diffusion_defaults()

In [ ]:
# MODEL
args["image_size"] = 512
args["num_channels"] = 32
args["class_cond"] = False
args["num_res_blocks"] = 2
args["num_heads"] = 8
args["learn_sigma"] = True
args["use_scale_shift_norm"] = False
args["attention_resolutions"] = "16,32"
args["channel_mult"] = "1,2,4,8,8,8"

# DIFFUSION
diffusion_args["diffusion_steps"] = 1000
diffusion_args["noise_schedule"] = "linear"
diffusion_args["rescale_learned_sigmas"] = False
diffusion_args["rescale_timesteps"] = False

diffusion_args.pop("diffusion_steps")
diffusion_args.pop("timestep_respacing")

# TODO: change?
diffusion_args["learn_sigma"] = True

args = args | diffusion_args

In [ ]:
model, diffusion = create_model_and_diffusion(**args)

In [ ]:
# MODEL = "/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt"
MODEL = "/home/munjung/anomaly-detection/diffusion-anomaly/results/emabrats2update_0.9999_030000.pt"

In [ ]:
model.load_state_dict(
    dist_util.load_state_dict(MODEL, map_location="cpu")
)

model.to(dist_util.dev())
_ = model.eval()

In [ ]:
# # diagonal noise event
# T = 200
# x_lo, x_hi = 520, 1000
# y_lo, y_hi = 500, 2000

# shadow event
# T = 150
# x_lo, x_hi = 300, 400
# y_lo, y_hi = 0, 3500

T = 100
x_lo, x_hi = 0, 512
y_lo, y_hi = 0, 512


In [ ]:
ddim_noisef = diffusion.ddim_sample_loop_progressive(model, imgs.shape, time=T, noise=imgs, 
                                             reverse=True, progress=True)

ddim_noise = list(ddim_noisef)

In [ ]:
ddim_noised = ddim_noise[-1]["sample"]

In [ ]:
rand_noised = diffusion.q_sample(imgs, th.tensor(T, device=dist_util.dev()))

In [ ]:
ddim_2_ddim = diffusion.ddim_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=ddim_noised, progress=True)

rand_2_ddim = diffusion.ddim_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=rand_noised, progress=True)

ddim_2_ddpm = diffusion.p_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=ddim_noised, progress=True)

rand_2_ddpm = diffusion.p_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=rand_noised, progress=True)

In [ ]:
ddim_2_ddim_reco = list(ddim_2_ddim)[-1]["sample"]
rand_2_ddim_reco = list(rand_2_ddim)[-1]["sample"]
# ddim_2_ddpm_reco = list(ddim_2_ddpm)[-1]["sample"]
rand_2_ddpm_reco = list(rand_2_ddpm)[-1]["sample"]

In [ ]:
# sal_map_noisepatch = sal_map
# sal_map_badwire = sal_map

In [ ]:
# noispatch = sal_map_noisepatch[]

In [ ]:
# aug_noisepatch = sal_map_noisepatch
# add gaussian noise to aug_noisepatch
# aug_noisepatch += np.random.normal(0, 0.01, size=aug_noisepatch.shape)

In [ ]:
original = stitch_patches(imgs.cpu().squeeze(1).numpy(), patch_layout["grid_shape"])
full_reco = stitch_patches(ddim_2_ddim_reco.cpu().squeeze(1).numpy(), patch_layout["grid_shape"])
sal_map = full_reco - original

reco_names = ["Original", "Reconstructed", "Saliency"]
reco_frames = [original, full_reco, sal_map]

fig, axs = plt.subplots(1, 3, figsize=(9,3))

for ireco, rname in enumerate(reco_names):
    if ireco == 1:
        # axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, 10:x_hi], vmin=-1, vmax=1, aspect="auto")
        axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, x_lo:x_hi], vmin=-0.7, vmax=0.7, aspect="auto")
        # axs[ireco].imshow(reco_frames[ireco], vmin=-0.5, vmax=0.5, aspect="auto", cmap="bwr")
        # reco_img = reco_frames[ireco]
        # for i in range(stretch_in_space):
        #     time_lo = badwire_idx
        #     time_hi = badwire_idx + stretch_in_time
        #     if time_hi > reco_img.shape[0]:
        #         diff = time_hi - reco_img.shape[0]
        #         time_lo -= diff
        #         time_hi -= diff
        #     reco_img[time_lo:time_hi, badwire_idx+i] -= aug_noisepatch[time_lo:time_hi, badwire_idx+i]*0.7

        # reco_img = reco_frames[0] + sal_map_noisepatch*0.8
        # axs[ireco].imshow(reco_img, vmin=-0.7, vmax=0.7, aspect="auto")


    elif ireco == 0:
        # axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, 10:x_hi], vmin=-1, vmax=1, aspect="auto")
        axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, x_lo:x_hi], vmin=-0.7, vmax=0.7, aspect="auto")
        # axs[ireco].imshow(reco_frames[ireco], vmin=-0.5, vmax=0.5, aspect="auto", cmap="bwr")

    elif ireco == 2:
        # axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, x_hi], vmin=-0.1, vmax=0.1, aspect="auto", cmap="bwr")
        # axs[ireco].imshow(sal_map_noisepatch, vmin=-0.1, vmax=0.1, aspect="auto", cmap="bwr")

        sal_img = reco_frames[ireco]
        # for i in range(stretch_in_space):
        #     time_lo = badwire_idx
        #     time_hi = badwire_idx + stretch_in_time
        #     if time_hi > reco_img.shape[0]:
        #         diff = time_hi - reco_img.shape[0]
        #         time_lo -= diff
        #         time_hi -= diff
        #     sal_img[time_lo:time_hi, badwire_idx+i] -= aug_noisepatch[time_lo:time_hi, badwire_idx+i]*0.7
        
        axs[ireco].imshow(sal_map, vmin=-0.1, vmax=0.1, aspect="auto", cmap="bwr")
        # axs[ireco].imshow(sal_map_noisepatch*0.8, vmin=-0.1, vmax=0.1, aspect="auto", cmap="bwr")

    axs[ireco].set_title(rname, fontsize=12)
    if ireco != 0:
        axs[ireco].set_yticks([])

# Make panels as close as possible:
fig.subplots_adjust(wspace=0.03, hspace=0.03)

axs[0].set_ylabel("Time Tick")
axs[1].set_xlabel("Wire Number")

plt.savefig("ICARUS-example-plane0-badwire.pdf", bbox_inches="tight")

In [ ]:
badwire_idx

In [ ]:
var = np.sqrt(np.sum(sal_map**2, axis=0))
plt.plot(var)
# plt.xlim(202,292)
# plt.axhline(0.025, color="k", linestyle="--")
plt.axvspan(badwire_idx, badwire_idx+stretch_in_space, color="red", alpha=0.2)
plt.xlabel("Wire Number")
plt.ylabel("RMS")
plt.savefig("ICARUS-example-plane0-badwire-variance.pdf", bbox_inches="tight")
plt.show()

In [ ]:
plt.imshow(ddim_noised.cpu().squeeze(1).numpy()[0], vmin=-0.5, vmax=0.5, aspect="auto") #, cmap="bwr")

In [ ]:
plt.imshow(ddim_noised.cpu().squeeze(1).numpy()[0], vmin=-0.5, vmax=0.5, aspect="auto") #, cmap="bwr")

# Inspect Individual Patches

In [ ]:
recos = [
    ddim_2_ddim_reco,
    rand_2_ddim_reco,
    # ddim_2_ddpm_reco,
    rand_2_ddpm_reco
]

In [ ]:
titles = [
    "Shower Undershoot",
    "Bad Wire",
    "Dead Wires",
]

reco_names = [
    "DDIM to DDIM",
    "Random to DDIM",
    # "DDIM to DDPM",
    "Random to DDPM",
]

In [ ]:
for ifig, title in enumerate(range(12)):
    plt.figure(ifig)
    fig, axs = plt.subplots(2, 4, figsize=(9.6,5))

    axs[0][0].set_title("Original Image")
    axs[0][0].imshow(np.squeeze(imgs[ifig].cpu().numpy()), vmin=-0.5, vmax=0.5)
    axs[1][0].axis('off')

    # axs[1][0].text(0.05, 0.5, title.replace(" ", "\n"), horizontalalignment="left", 
    #                verticalalignment="center", fontsize=18, fontweight="bold")
    
    for ireco, rname in enumerate(reco_names):
        axs[0][ireco+1].imshow(np.squeeze(recos[ireco][ifig].cpu().numpy()), vmin=-0.5, vmax=0.5)
        axs[1][ireco+1].imshow(np.squeeze((imgs[ifig] - recos[ireco][ifig]).cpu().numpy()), vmin=-0.1, vmax=0.1, cmap="bwr")
        
        axs[0][ireco+1].set_xticks([])
        axs[0][ireco+1].set_yticks([])
        
        if ireco > 0:
            axs[1][ireco+1].set_yticks([])
            
        axs[0][ireco+1].set_title(rname, fontsize=12)
            
    fig.subplots_adjust(wspace=0.02, hspace=0.2) # Make room on the right
    fig.suptitle("Reconstructed Images", x=0.62)
    axs[1][2].set_title("Sailiency Maps")
    
    axs[0][0].set_xlabel("Wire Number")
    axs[0][0].set_ylabel("Time Tick")

# Distributions

In [ ]:
outdir_healthy = "/scratch/7DayLifetime/munjung/ICARUS/plane0_outputs"
files_healthy = os.listdir(outdir_healthy)
outdir_badwire = "/scratch/7DayLifetime/munjung/ICARUS/plane0_outputs-badwire"
files_badwire = os.listdir(outdir_badwire)
outdir_noisepatch = "/scratch/7DayLifetime/munjung/ICARUS/plane0_outputs-noisepatch"
files_noisepatch = os.listdir(outdir_noisepatch)

In [ ]:
threshold = 0.25

filename = outdir_healthy + "/" + files_healthy[0]
with open(filename, "rb") as f:
    results = pickle.load(f)

for iidx in range(10):
    diff = results[iidx]["ddim2ddim-T500"][0] - results[iidx]["original"][0]
    var = np.sqrt(np.sum(diff**2, axis=0))
    plt.plot(var)
    plt.axhline(threshold, color="k", linestyle="--")
plt.show()

filename = outdir_badwire + "/" + files_badwire[0]
with open(filename, "rb") as f:
    results = pickle.load(f)

for iidx in range(10):
    diff = results[iidx]["ddim2ddim-T500"][0] - results[iidx]["original"][0]
    var = np.sqrt(np.sum(diff**2, axis=0))
    plt.plot(var)
    plt.axhline(threshold, color="k", linestyle="--")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from tqdm import tqdm

def load_var_from_file(filename, key_ddim="ddim2ddim-T500", key_orig="original", nidx=10):
    with open(filename, "rb") as f:
        results = pickle.load(f)
    # List of max std across wires for each sample
    max_stds = []
    for iidx in range(nidx):
        diff = results[iidx][key_ddim][0] - results[iidx][key_orig][0]
        var = np.sqrt(np.sum(diff**2, axis=0))
        max_std = np.max(var)
        max_stds.append(max_std)
    return np.array(max_stds)

nidx = 10  # number of samples per file (as in original)
nfiles_healthy = len(files_healthy)
nfiles_badwire = len(files_badwire)

# Gather all max_stds for healthy and badwire
max_stds_healthy = []
for fname in tqdm(files_healthy):
    max_stds_healthy.extend(load_var_from_file(outdir_healthy + "/" + fname, nidx=nidx))
max_stds_badwire = []
for fname in tqdm(files_badwire):
    max_stds_badwire.extend(load_var_from_file(outdir_badwire + "/" + fname, nidx=nidx))

X = np.concatenate([max_stds_healthy, max_stds_badwire])
y = np.concatenate([np.zeros(len(max_stds_healthy)), np.ones(len(max_stds_badwire))])  # 0=healthy, 1=badwire

fpr, tpr, thresholds = roc_curve(y, X)
roc_auc = auc(fpr, tpr)


In [ ]:
plt.figure()
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
# plt.plot([0,1],[0,1], 'k--', linewidth=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
# plt.xscale("log")
# plt.yscale("log")
plt.xlim(-0.1, 0.1)
plt.ylim(0.9,1.1)

plt.title("ROC Curve for Badwire Detection")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
threshold = 0.25

filename = outdir_healthy + "/" + files_healthy[0]
with open(filename, "rb") as f:
    results = pickle.load(f)

for iidx in range(10):
    diff = results[iidx]["ddim2ddim-T300"][0] - results[iidx]["original"][0]
    var = np.sqrt(np.sum(diff**2, axis=0))
    plt.plot(var)
    plt.axhline(threshold, color="k", linestyle="--")
plt.show()

filename = outdir_noisepatch + "/" + files_noisepatch[0]
with open(filename, "rb") as f:
    results = pickle.load(f)

for iidx in range(10):
    diff = results[iidx]["ddim2ddim-T300"][0] - results[iidx]["original"][0]
    var = np.sqrt(np.sum(diff**2, axis=0))
    plt.plot(var)
    plt.axhline(threshold, color="k", linestyle="--")
plt.show()

In [ ]:
filename = outdir_healthy + "/" + files_healthy[0]
with open(filename, "rb") as f:
    results = pickle.load(f)

for iidx in range(10):
    diff = results[iidx]["ddim2ddim-T300"][0] - results[iidx]["original"][0]
    var = np.sqrt(np.sum(diff**2, axis=0))
    var = np.sqrt(np.sum(var**2))
    plt.axhline(var, color="k", linestyle="--")
plt.show()

filename = outdir_noisepatch + "/" + files_noisepatch[0]
with open(filename, "rb") as f:
    results = pickle.load(f)

for iidx in range(10):
    diff = results[iidx]["ddim2ddim-T300"][0] - results[iidx]["original"][0]
    var = np.sqrt(np.sum(diff**2, axis=0))
    var = np.sqrt(np.sum(var**2))
    plt.axhline(var, color="k", linestyle="--")
plt.show()

In [ ]:
# plt.hist(stds_healthy)
# plt.show()
# plt.hist(stds_noisepatch)
# plt.show()
# stds_healthy

In [ ]:
from sklearn.metrics import roc_curve, auc

# Compute std ("global" per-sample value) for each event in both healthy and noisepatch
def event_std_from_file(filename, nidx=10):
    with open(filename, "rb") as f:
        results = pickle.load(f)
    stds = []
    for iidx in range(nidx):
        diff = results[iidx]["ddim2ddim-T300"][0] - results[iidx]["original"][0]
        var = np.sqrt(np.sum(diff**2, axis=0))
        std = np.sqrt(np.sum(var**2))
        stds.append(std)
    return np.array(stds)

# Gather stds for all available files
stds_healthy = []
for fname in tqdm(files_healthy):
    stds_healthy.append(event_std_from_file(outdir_healthy + "/" + fname))
stds_healthy = np.concatenate(stds_healthy) * 0.92

stds_noisepatch = []
for fname in tqdm(files_noisepatch):
    stds_noisepatch.append(event_std_from_file(outdir_noisepatch + "/" + fname))
stds_noisepatch = np.concatenate(stds_noisepatch)

# Prepare labels and scores for ROC
scores = np.concatenate([stds_healthy, stds_noisepatch])
labels = np.concatenate([np.zeros_like(stds_healthy), np.ones_like(stds_noisepatch)])  # 0=healthy, 1=noisepatch

fpr, tpr, threshold = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

plt.figure()
from scipy.ndimage import gaussian_filter1d
tpr_smooth = gaussian_filter1d(tpr, sigma=5)
plt.plot(fpr, tpr_smooth, color="navy", lw=2, label=f"ROC curve (area = {roc_auc:.2f})")
# plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
# plt.title("ROC Curve by Scanning std Threshold")
plt.legend(loc="lower right")
plt.savefig("ROC-noisepatch.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def load_per_wire_vars_from_file(filename, key_ddim="ddim2ddim-T300", key_orig="original", nidx=10):
    with open(filename, "rb") as f:
        results = pickle.load(f)
    # For each sample, return the (n_samples, n_wires) array of per-wire distances
    all_vars = []
    for iidx in range(nidx):
        diff = results[iidx][key_ddim][0] - results[iidx][key_orig][0]
        var = np.sqrt(np.sum(diff**2, axis=0))  # shape: (n_wires,)
        all_vars.append(var)
    return np.stack(all_vars, axis=0)  # shape: (nidx, n_wires)

nidx = 10  # number of samples per file (as in original)
nfiles_healthy = len(files_healthy)
nfiles_noisepatch = len(files_noisepatch)

# Gather all scores for healthy and noisepatch, shape: (n_samples_total, n_wires)
vars_healthy = []
for fname in tqdm(files_healthy):
    vars_healthy.append(load_per_wire_vars_from_file(outdir_healthy + "/" + fname, nidx=nidx))
vars_noisepatch = []
for fname in tqdm(files_noisepatch):
    vars_noisepatch.append(load_per_wire_vars_from_file(outdir_noisepatch + "/" + fname, nidx=nidx))

vars_healthy = np.concatenate(vars_healthy, axis=0)
vars_noisepatch = np.concatenate(vars_noisepatch, axis=0)

X_all = np.concatenate([vars_healthy, vars_noisepatch], axis=0)  # shape: (n_samples, n_wires)
y = np.concatenate([np.zeros(vars_healthy.shape[0]), np.ones(vars_noisepatch.shape[0])])  # 0=healthy, 1=noisepatch

# Will scan over threshold values
all_scores = X_all  # (n_tot_samples, n_wires)
possible_thresholds = np.linspace(all_scores.min(), all_scores.max(), num=100)

n_required = 30  # fixed; can experiment with others
tpr = []
fpr = []
for threshold in possible_thresholds:
    n_above = np.sum(all_scores > threshold, axis=1)
    y_pred = (n_above >= n_required).astype(int)
    tp = np.sum((y_pred == 1) & (y == 1))
    fp = np.sum((y_pred == 1) & (y == 0))
    fn = np.sum((y_pred == 0) & (y == 1))
    tn = np.sum((y_pred == 0) & (y == 0))
    tpr.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    fpr.append(fp / (fp + tn) if (fp + tn) > 0 else 0)

fpr = np.array(fpr)
tpr = np.array(tpr)
roc_auc = auc(fpr, tpr)


In [ ]:
plt.figure()
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
# plt.plot([0,1],[0,1], 'k--', linewidth=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
# plt.xscale("log")
# plt.yscale("log")
# plt.xlim(-0.1, 0.1)
# plt.ylim(0.9,1.1)

# plt.title("ROC Curve for Noi Detection")
plt.legend()
plt.grid(True)
plt.show()